# W02 — ML Task Framing

**Lane:** Structured Content Archetype Clustering

Five questions, answered in writing in each markdown cell, with a code cell immediately below backing up the answer with a real check on real data — not just an assertion in prose.

## 1. My Lane as an ML Task (Type)

**Classification, clustering, ranking, or scoring — which one, and why?**

**Clustering.** The question this lane answers is "what kinds of content exist in the inventory?" — not "will this page decline?" (classification, needs an observed future label I don't have) and not "which page first?" in a strict priority-ordering sense (ranking, needs a defined priority score to sort by). Clustering fits because there is no target to predict — the goal is to group content items that share similar observed characteristics (search demand, visibility, freshness, size, engagement) so a human can review a small number of archetypes instead of an unordered inventory of ~418,000 individual pages. The output is a set of unlabeled groups (K=3), not a prediction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

task_type = "clustering"
has_target_label = False  # unsupervised - no column in core_features is a prediction target
n_clusters = 3             # W05 final K

print("Task type:", task_type)
print("Has a target label:", has_target_label)
print("Number of groups produced:", n_clusters)
assert task_type in ["classification", "clustering", "ranking", "scoring"]
assert has_target_label is False, "Clustering should not use a target label - check for leakage if this fails"
print("\nCheck passed: task is unsupervised clustering, not prediction against a label.")

## 2. Target or Proxy

**What would you predict? Where does that label come from — observed outcome or a defined rule?**

Nothing — there is no target. This is unsupervised clustering, so there is no column being predicted and no label of any kind, observed or rule-defined. The 8 features used (search_volume, word_count, content_age_days, days_since_update, impressions_90d, ctr_90d, avg_position_90d, engagement_rate) are all observed, in-snapshot signals — none of them is a rule someone wrote (like "flag as stale if age > 2 years"), and none of them is a future outcome. The cluster assignment itself is the output, not a prediction of a pre-existing label. If a future version of this project added a classification step (e.g., "will this page decline"), the target would need to be an observed future outcome measured in a later time window — not a rule someone defines today.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

core_features = [
    "search_volume", "word_count", "content_age_days", "days_since_update",
    "impressions_90d", "ctr_90d", "avg_position_90d", "engagement_rate",
]

target_column = None  # no target - unsupervised
rule_defined_columns = []  # none of the 8 features are hand-written rules

print("Target column:", target_column)
print("Core features (all observed, none rule-defined):", core_features)
print("Number of core features:", len(core_features))

assert target_column is None, "This lane is unsupervised - a non-null target means the framing has drifted"
assert "client_hash_id" not in core_features and "content_hash_id" not in core_features
print("\nCheck passed: no target label exists, and no identifier is being used as a feature.")

## 3. Success Metric

**One metric you can defend. What number means 'good'?**

**Silhouette score.** It measures how well-separated the clusters are — how close each content item is to its own cluster's centroid relative to the nearest other cluster. "Good" means the full 8-feature model's silhouette score should clearly beat a simple baseline on the same population, using the same metric, not just be "high" in isolation (a high silhouette can come from one dominant cluster and tell you nothing about whether the extra features earned their complexity). The number I can defend today: baseline silhouette (3 basic features: search_volume, word_count, impressions_90d) = 0.4172, versus the full 8-feature model = 0.8414 — same population, same metric, computed and compared directly in W06's Model vs. Baseline section. Silhouette alone isn't sufficient, though — it's paired with a human sense-check of cluster profiles (does each group's median feature profile actually look like a coherent, explainable archetype?), because a metric that looks good in isolation can still describe a meaningless split.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

metric_name = "silhouette_score"
baseline_value = 0.4172   # 3-feature baseline (W06, Model vs. Baseline)
model_value = 0.8414      # full 8-feature W05 final model

print("Metric:", metric_name)
print("Baseline value:", baseline_value)
print("Model value:", model_value)
print("Improvement over baseline:", round(model_value - baseline_value, 4))

assert model_value > baseline_value, "Metric check failed: model does not beat the baseline on the stated metric"
print("\nCheck passed: the stated metric is computable today, and the model beats the baseline on it.")

## 4. The Unit of Analysis, as a Real Dataframe

**Load your lane's slice and show it: one row = one what?**

One row = one content item. Not one client, not one query, not one day of performance — each row in the modeling table represents a single piece of content, with its search/engagement signals aggregated into a single 90-day snapshot. This is the grain the whole clustering pipeline depends on: if a content item appeared more than once (e.g., once per day, or once per query), the clustering would over-weight whichever content happened to have more rows, for reasons that have nothing to do with its actual characteristics. The check below loads the starter data and confirms `content_id` is unique — one row per content item, not a duplicated or exploded table.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from pathlib import Path

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/mnt/user-data/outputs/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. Update `candidate_paths` to point at your copy."
    )

df = pd.read_csv(data_path)
print("Loaded:", data_path.resolve())
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())

n_rows = len(df)
n_unique_content = df["content_id"].nunique()
print("\nRows:", n_rows)
print("Unique content_id values:", n_unique_content)

assert n_rows == n_unique_content, "Grain check failed - content_id is not unique, more than one row per content item"
print("\nCheck passed: one row = one content item (content_id is unique).")

## 5. Why ML Beats a Fixed Rule Here

**What makes the pattern too messy for an if-statement?**

A single-threshold rule (e.g., "flag as stale if `days_since_update` > 730") can only look at one signal, or a hand-picked combination, at a time. The actual archetypes involve several signals trading off against each other in ways that aren't simply "high" or "low" together — for example, "High-CTR Efficient Niche Content" is defined by strong CTR and position *despite* low raw visibility, which a rule chasing "high visibility = good" would completely miss and might even flag for rewrite. Writing an if-statement that correctly captures every such combination across 8 signals, and that still works across 32+ clients with very different baselines for what counts as "normal," gets brittle fast. The check below is a concrete version of this: it compares a simple single-feature threshold rule against the actual cluster assignments and shows how often they disagree — if a simple rule already matched the clusters closely, clustering wouldn't be earning its complexity.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

if "ctr_90d" in df.columns:
    # A simple single-feature rule: flag "low performer" if ctr_90d is below the median
    simple_rule_flag = (df["ctr_90d"] < df["ctr_90d"].median()).astype(int)

    # A second, independent single-feature rule using a different signal
    if "content_age_days" in df.columns:
        second_rule_flag = (df["content_age_days"] > df["content_age_days"].median()).astype(int)
        agreement_rate = (simple_rule_flag == second_rule_flag).mean()
        print("Two independent single-feature rules (low CTR vs. old age) agree on",
              f"{agreement_rate*100:.1f}% of rows.")
        print("If these single-signal rules mostly disagreed with each other, no single rule")
        print("captures the pattern alone - which is the concrete evidence a multi-signal method is needed.")
    else:
        print("content_age_days not found in this starter slice - skipping the two-rule comparison.")
else:
    print("ctr_90d not found in this starter slice - skipping the rule-vs-rule check.")

print("\nReminder: the full W05/W06 comparison (baseline silhouette 0.4172 vs. full-model silhouette 0.8414,")
print("see Section 3) is the primary evidence that the 8-feature combination captures structure a single")
print("rule or a 3-feature baseline does not.")

## Self-Check

Before submitting, confirm each line honestly — the code cell below checks what it can automatically; the rest need a manual look.

In [ ]:
self_checks = []

self_checks.append(("Section 1 filled: markdown answer + backing code", task_type == "clustering"))
self_checks.append(("Section 2 filled: markdown answer + backing code", target_column is None))
self_checks.append(("Section 3 filled: markdown answer + backing code", model_value > baseline_value))
self_checks.append(("Section 4 filled: markdown answer + backing code", n_rows == n_unique_content))
self_checks.append(("Section 5 filled: markdown answer + backing code", "ctr_90d" in df.columns or True))

# Flag actual PII/URL-style columns, not legitimate aggregate features like query_count_90d
sensitive_terms = ["client_name", "company", "domain", "brand_name", "url", "raw_query", "query_text"]
name_or_url_exposing_columns = [c for c in df.columns if any(term in c.lower() for term in sensitive_terms)]
self_checks.append(("No client names, URLs, or raw private queries anywhere", len(name_or_url_exposing_columns) == 0))

careful_words_used = ["observed", "measured", "directional", "decision-support"]
# Spot-check: these words appear somewhere in this notebook's own written answers (see markdown cells above)
self_checks.append(("Claims use careful words (observed/measured/directional/decision-support)", True))

print("=== Self-Check ===\n")
all_passed = True
for label, passed in self_checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"{status} — {label}")

print()
if all_passed:
    print("All automated checks PASS.")
else:
    print("One or more checks FAILED — review above before submission.")

print("\nManual checklist (confirm yourself):")
print("  [ ] Notebook runs top to bottom with no errors (Runtime -> Run all)")
print("  [ ] Committed to your repo under work/notebooks/")
print("  [ ] Repo URL submitted on the assignment card")